# Week 2: Data Preparation & EDA
## Spam Classifier Project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('stopwords')

## Step 1: Load and Clean Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/spam.csv', encoding='latin-1')
print(f"Original shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst rows:")
print(df.head())

In [ ]:
# Clean data
df = df[['v1', 'v2']]
df.columns = ['target', 'text']
df['target'] = df['target'].map({'ham': 0, 'spam': 1})
df = df.drop_duplicates(keep='first')

print(f"Cleaned shape: {df.shape}")
print(f"\nTarget distribution:\n{df['target'].value_counts()}")

## Step 2: Extract Meta-Features

In [ ]:
# Extract meta-features
df['num_characters'] = df['text'].apply(len)
df['num_words'] = df['text'].apply(lambda x: len(x.split()))
df['num_sentences'] = df['text'].apply(lambda x: len(x.split('.')))

print("Meta-features extracted")
print(df[['target', 'num_characters', 'num_words', 'num_sentences']].head())

## Step 3: Visualizations

In [ ]:
# Create visualizations directory
import os
os.makedirs('../visualizations', exist_ok=True)

# 1. Class distribution
fig, ax = plt.subplots(figsize=(8, 6))
df['target'].value_counts().plot(kind='pie', labels=['Ham', 'Spam'], autopct='%1.1f%%', ax=ax)
plt.title('Class Distribution')
plt.ylabel('')
plt.tight_layout()
plt.savefig('../visualizations/01_class_distribution.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ 01_class_distribution.png saved")

In [ ]:
# 2. Distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df[df['target']==0]['num_characters'], bins=50, alpha=0.7, label='Ham', color='green')
axes[0].hist(df[df['target']==1]['num_characters'], bins=50, alpha=0.7, label='Spam', color='red')
axes[0].set_xlabel('Number of Characters')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Character Distribution')
axes[0].legend()

axes[1].hist(df[df['target']==0]['num_words'], bins=50, alpha=0.7, label='Ham', color='green')
axes[1].hist(df[df['target']==1]['num_words'], bins=50, alpha=0.7, label='Spam', color='red')
axes[1].set_xlabel('Number of Words')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Word Count Distribution')
axes[1].legend()

axes[2].hist(df[df['target']==0]['num_sentences'], bins=50, alpha=0.7, label='Ham', color='green')
axes[2].hist(df[df['target']==1]['num_sentences'], bins=50, alpha=0.7, label='Spam', color='red')
axes[2].set_xlabel('Number of Sentences')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Sentence Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig('../visualizations/02_distributions.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ 02_distributions.png saved")

In [ ]:
# 3. Correlation heatmap
correlation_data = df[['target', 'num_characters', 'num_words', 'num_sentences']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_data, annot=True, cmap='coolwarm', center=0, fmt='.3f', 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Pearson Correlation Heatmap')
plt.tight_layout()
plt.savefig('../visualizations/03_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ 03_correlation_heatmap.png saved")

## Step 4: NLP Preprocessing Pipeline

In [ ]:
# 5-step preprocessing function
def preprocess_text(text):
    # 1. Lowercasing
    text = text.lower()
    
    # 2. Tokenization
    tokens = word_tokenize(text)
    
    # 3. Special character removal
    tokens = [token for token in tokens if token.isalnum()]
    
    # 4. Stopword removal
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    
    # 5. Stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(token) for token in tokens]
    
    return ' '.join(tokens)

# Apply preprocessing
df['cleaned_text'] = df['text'].apply(preprocess_text)
print("✓ Preprocessing complete")
print(f"\nExample:")
print(f"Original: {df['text'].iloc[0][:80]}...")
print(f"Cleaned:  {df['cleaned_text'].iloc[0][:80]}...")

## Step 5: Word Clouds

In [ ]:
# 4. Word clouds
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Ham word cloud
ham_text = ' '.join(df[df['target']==0]['cleaned_text'])
wordcloud_ham = WordCloud(width=400, height=300, background_color='white', colormap='Greens').generate(ham_text)
axes[0].imshow(wordcloud_ham, interpolation='bilinear')
axes[0].set_title('Word Cloud - Ham (Legitimate)')
axes[0].axis('off')

# Spam word cloud
spam_text = ' '.join(df[df['target']==1]['cleaned_text'])
wordcloud_spam = WordCloud(width=400, height=300, background_color='white', colormap='Reds').generate(spam_text)
axes[1].imshow(wordcloud_spam, interpolation='bilinear')
axes[1].set_title('Word Cloud - Spam')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('../visualizations/04_word_clouds.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ 04_word_clouds.png saved")

In [ ]:
# 5. Top 30 words
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 30 ham words
ham_words = Counter(' '.join(df[df['target']==0]['cleaned_text']).split())
ham_top30 = dict(ham_words.most_common(30))
axes[0].barh(list(ham_top30.keys())[::-1], list(ham_top30.values())[::-1], color='green')
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 30 Words in Ham Messages')
axes[0].invert_yaxis()

# Top 30 spam words
spam_words = Counter(' '.join(df[df['target']==1]['cleaned_text']).split())
spam_top30 = dict(spam_words.most_common(30))
axes[1].barh(list(spam_top30.keys())[::-1], list(spam_top30.values())[::-1], color='red')
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top 30 Words in Spam Messages')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../visualizations/05_top_words.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ 05_top_words.png saved")

## Step 6: Save Cleaned Data

In [ ]:
# Save cleaned data
df.to_csv('../data/cleaned_data.csv', index=False)
print(f"✓ Cleaned data saved")
print(f"\nFinal dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")